# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mol_instruction_dataset = datasets.load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)
qm9_data = mol_instruction_dataset['property_prediction']
qm9_homo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo
            )
qm9_lumo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_lumo
            )
qm9_homo_lumo_gap_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo_lumo_gap
            )

In [3]:
def get_qm9_data_list(
        qm9_data,
        task,
        instruction_templates,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(qm9_data)))
    for i in iter_bar:
        data_instance = qm9_data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [4]:


def get_idx_with_abs_distance(idxs, pos, distance):
    idxs = np.array(idxs)
    larger_idxs = idxs[idxs > pos + distance]
    smaller_idxs = idxs[idxs < pos - distance]
    # get union of larger and smaller idxs
    union_idx = np.union1d(larger_idxs, smaller_idxs)
    return union_idx

def get_molpo_data_list(
        qm9_data,
        task,
        instruction_templates,
        num_bins=10,
        num_reject=50,
        tag="target_value_0414",
    ):
    list_train_mol = []
    list_train_label = []

    list_test_mol = []
    list_test_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(qm9_data)))
    for i in iter_bar:
        data_instance = qm9_data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_train_label.append(label)
            list_train_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_test_label.append(label)
            list_test_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    # change str to float
    list_train_float_label = [float(i) for i in list_train_label]
    list_test_float_label = [float(i) for i in list_test_label]

    # sort the idx by the float label
    train_idx = np.argsort(list_train_float_label)
    list_train_mol = [list_train_mol[i] for i in train_idx]
    list_train_label = [list_train_label[i] for i in train_idx]
    list_train_float_label = [list_train_float_label[i] for i in train_idx]

    test_idx = np.argsort(list_test_float_label)
    list_test_mol = [list_test_mol[i] for i in test_idx]
    list_test_label = [list_test_label[i] for i in test_idx]
    list_test_float_label = [list_test_float_label[i] for i in test_idx]


    # devide train_idx into num_bins
    bin_list_train_mol = np.array_split(list_train_mol, num_bins)
    bin_list_train_label = np.array_split(list_train_float_label, num_bins)

    bin_list_test_mol = np.array_split(list_test_mol, num_bins)
    bin_list_test_label = np.array_split(list_test_float_label, num_bins)

    bin_list_reject_mol = []
    for i in range(num_bins):
        list_reject_mol_bins = get_idx_with_abs_distance(range(num_bins), i, 2)

        # concatenate the bins
        concat_reject_mol = []
        for j in list_reject_mol_bins:
            concat_reject_mol += bin_list_train_mol[j].tolist()
        bin_list_reject_mol.append(concat_reject_mol)    

    bin_list_tr_data = []
    bin_list_te_data = []
    for i in range(num_bins):
        list_reject_mol = bin_list_reject_mol[i]
        np.random.shuffle(list_reject_mol)
        list_reject_mol = list_reject_mol[:num_reject]

        list_tr_data = get_data_list(
            list_mol=bin_list_train_mol[i],
            list_label=bin_list_train_label[i],
            task=task,
            instruction_templates=instruction_templates,
            list_reject_mol=list_reject_mol,
            num_reject=num_reject,
        )
        bin_list_tr_data.append(list_tr_data)

        list_te_data = get_data_list(
            list_mol=bin_list_test_mol[i],
            list_label=bin_list_test_label[i],
            task=task,
            instruction_templates=instruction_templates,
            list_reject_mol=list_reject_mol,
            num_reject=num_reject,
        )
        bin_list_te_data.append(list_te_data)

    # concatenate the bins
    list_train_data = []
    for i in range(num_bins):
        list_train_data += bin_list_tr_data[i]
        
    list_test_data = []
    for i in range(num_bins):
        list_test_data += bin_list_te_data[i]
    print(len(list_train_data), len(list_test_data))

    data_dict = {
        "train": list_train_data,
        "test": list_test_data
    }

    for split in ["train", "test"]:
        list_data = data_dict[split]
        
        dataset = datasets.Dataset.from_list(list_data)
        dataset.save_to_disk(
            f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_{tag}_molpo"
        )

In [6]:
get_molpo_data_list(
    qm9_data=qm9_homo_data,
    task="qm9_homo",
    instruction_templates=instructions_smol.qm9_homo,
    num_bins=10,
    num_reject=50,
)

100%|██████████| 68/68 [00:00<00:00, 112.22it/s]


120062 684


Saving the dataset (1/1 shards): 100%|██████████| 684/684 [00:00<00:00, 2427.50 examples/s]
